# Decision Trees at Large Scale

## Learning Objectives

1. **Recall** how standard CART builds decision trees
2. **Explain** how SLIQ/SPRINT use disk-friendly sorted attribute lists
3. **Implement** histogram-based split finding for large datasets
4. **Describe** how random forests are parallelized in MapReduce

## Standard CART: Memory Bottleneck

**CART** (Classification and Regression Trees) finds the best split at each node:
$$\text{split}^* = \arg\max_{j,t} \text{Gain}(S, j, t) = H(S) - \frac{|S_L|}{|S|}H(S_L) - \frac{|S_R|}{|S|}H(S_R)$$

**At scale (large $n$):**
- Sorting all features per node: $O(n d \log n)$ per tree level
- Requires all data in memory: not feasible for $n > 10^8$

**Solutions:**
1. **SLIQ/SPRINT:** pre-sort attribute lists; use disk efficiently
2. **Histogram methods:** discretize features into bins; approximate best split
3. **XGBoost/LightGBM:** weighted quantile sketch + histogram approach

In [1]:
import math
from collections import Counter

def gini(labels):
    """Gini impurity."""
    n = len(labels)
    if n == 0: return 0.0
    cnt = Counter(labels)
    return 1.0 - sum((v/n)**2 for v in cnt.values())

def entropy(labels):
    n = len(labels)
    if n == 0: return 0.0
    cnt = Counter(labels)
    return -sum((v/n)*math.log2(v/n) for v in cnt.values() if v > 0)

def best_split_exact(X, y, feature_idx):
    """Find best split threshold by sorting (standard CART)."""
    n = len(X)
    vals = sorted(set(X[i][feature_idx] for i in range(n)))
    best_gain, best_t = -1, None
    h_total = entropy(y)
    for t in vals[:-1]:
        left  = [y[i] for i in range(n) if X[i][feature_idx] <= t]
        right = [y[i] for i in range(n) if X[i][feature_idx] >  t]
        gain = h_total - len(left)/n*entropy(left) - len(right)/n*entropy(right)
        if gain > best_gain:
            best_gain, best_t = gain, t
    return best_t, best_gain

def best_split_histogram(X, y, feature_idx, n_bins=10):
    """Histogram-based approximate split finding."""
    n = len(X)
    col = [X[i][feature_idx] for i in range(n)]
    vmin, vmax = min(col), max(col)
    bin_width = (vmax - vmin) / n_bins if vmax > vmin else 1
    # Accumulate per-bin statistics
    bin_left = [[] for _ in range(n_bins)]  # labels in each bin
    for i in range(n):
        b = min(int((col[i]-vmin)/bin_width), n_bins-1)
        bin_left[b].append(y[i])
    # Sweep thresholds
    h_total = entropy(y)
    left_acc, right_acc = [], list(y)
    best_gain, best_t = -1, None
    for b in range(n_bins-1):
        left_acc.extend(bin_left[b])
        right_acc = [lbl for lbl in right_acc if lbl not in set(bin_left[b])]
        # Recompute right_acc properly
        right_acc = []
        for bb in range(b+1, n_bins): right_acc.extend(bin_left[bb])
        if not left_acc or not right_acc: continue
        gain = h_total - len(left_acc)/n*entropy(left_acc) - len(right_acc)/n*entropy(right_acc)
        if gain > best_gain:
            best_gain, best_t = gain, vmin + (b+1)*bin_width
    return best_t, best_gain

import random
rng = random.Random(7)
n, d = 500, 3
X = [[rng.gauss(0,1) for _ in range(d)] for _ in range(n)]
y = [1 if X[i][0]+X[i][1] > 0 else 0 for i in range(n)]

for feat in range(d):
    t_exact, g_exact = best_split_exact(X, y, feat)
    t_hist,  g_hist  = best_split_histogram(X, y, feat, n_bins=20)
    print(f"Feature {feat}: exact(t={t_exact:.3f}, gain={g_exact:.4f}) | "
          f"hist(t={t_hist:.3f}, gain={g_hist:.4f})")

Feature 0: exact(t=-0.559, gain=0.2094) | hist(t=-0.515, gain=0.1997)
Feature 1: exact(t=-0.018, gain=0.2250) | hist(t=0.151, gain=0.2081)
Feature 2: exact(t=2.395, gain=0.0073) | hist(t=2.397, gain=0.0073)


## SLIQ and SPRINT

**SLIQ (Mehta et al. 1996):**
- Pre-sort each attribute into a sorted list `[(val, instance_id)]` — done once, stored on disk
- During training, each node maintains a class list `[class_label]` indexed by instance_id
- To find the best split: scan sorted attribute list, updating left/right label counts
- No sorting per node → $O(n)$ per split evaluation

**SPRINT (Shafer et al. 1996):**
- Each attribute list is self-contained: `[(val, instance_id, class_label)]`
- No shared class list → can be fully parallelized across processors
- After a split, partition each attribute list independently

**Both:** require $O(nd)$ preprocessing, but split finding is $O(n)$ not $O(n \log n)$.

## Random Forests in MapReduce

**Single random forest tree:** easily parallelized.

**MapReduce approach:**
- **Map:** each mapper trains one tree on a random subset of features and bootstrap sample
- **Reduce:** collect all $T$ trees; ensemble prediction = majority vote / mean

**Feature-parallel alternative:**
- **Map:** each mapper evaluates all thresholds for one feature
- **Reduce:** aggregate per-feature gains; pick the best split globally

**Scale:** XGBoost and LightGBM use the histogram + split-finding approach with GPU acceleration, enabling $n = 10^9$ and $d = 10^6$ in practice.